In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Lectura de Datos

Leemos el archivo que contiene los datos de entrenamiento. Convertimos las variables categóricas binarias en numéricas para facilitar su análisis.

In [2]:
df_hoteles = pd.read_csv('../../data/inputs/hoteles-entrena.csv')


In [3]:
df_hoteles['hotel'].unique()

array(['Resort_Hotel', 'City_Hotel'], dtype=object)

In [4]:
#Convertimos la variable children a numérica
df_hoteles['children'] = df_hoteles['children'].map({'children': 1, 'none': 0})
#Convertimos la variable hotel a numérica
df_hoteles['hotel'] = df_hoteles['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0})
 

#### Valores faltantes


In [5]:
# Valores faltantes

#Completar nulos en country
#Nulos en country
df_hoteles['country'] = df_hoteles['country'].fillna('OTROS')

#Nulos en 'agent
df_hoteles['agent'] = df_hoteles['agent'].fillna(0)
#Nulos en 'company
df_hoteles['company'] = df_hoteles['company'].fillna(0)

In [6]:
#Revisamos los nulos restantes
nulos = df_hoteles.isnull().sum()
if (nulos.sum() == 0):
    print('No hay valores nulos en el DataFrame.')
else:
    print(f'Total de Nulos por Variable:\n{df_hoteles.isnull().sum()}')

No hay valores nulos en el DataFrame.


In [7]:
# Contar reservas por país
country_counts = df_hoteles['country'].value_counts()

# Calcular porcentaje sobre el total
country_percent = (country_counts / country_counts.sum()) * 100

# Crear dataframe con top 10
top10_countries = pd.DataFrame({
    'country': country_counts.index[:10],
    'count': country_counts.values[:10],
    'percentage': country_percent.values[:10]
})

print(top10_countries)
print(f"Top 10 representan el {country_percent[:10].sum():.2f}% de todas las reservas.")


  country  count  percentage
0     PRT  14909   28.140277
1     GBR   6817   12.866877
2     FRA   5949   11.228554
3     ESP   4523    8.537023
4     DEU   4268    8.055718
5     IRL   1759    3.320058
6     ITA   1696    3.201148
7     BEL   1301    2.455597
8     NLD   1180    2.227214
9     USA   1124    2.121515
Top 10 representan el 82.15% de todas las reservas.


In [8]:
#Convertimos todos los países que no están en el top 10 a 'OTROS'
top10_list = top10_countries['country'].tolist()
df_hoteles['country'] = df_hoteles['country'].apply(lambda x: x if x in top10_list else 'OTROS')
df_hoteles['country'].unique()


array(['PRT', 'GBR', 'IRL', 'OTROS', 'ESP', 'USA', 'DEU', 'BEL', 'FRA',
       'ITA', 'NLD'], dtype=object)

#### Modificación de la variable sobre la fecha de llegada

Vamos a crear nuevas variables a partir de la variable 'arrival_date' para facilitar el análisis y modelado. Separamos en año, mes y día, pero mes y día lo hacemos en forma radial para capturar mejor la naturaleza cíclica de estas variables.

In [9]:
#Convertir mes de llegada a variables cíclicas
df_hoteles['arrival_date'] = pd.to_datetime(df_hoteles['arrival_date'], format='%Y-%m-%d')
df_hoteles['arrival_month_sin'] = np.sin(2 * np.pi * df_hoteles['arrival_date'].dt.month / 12)
df_hoteles['arrival_month_cos'] = np.cos(2 * np.pi * df_hoteles['arrival_date'].dt.month / 12)
df_hoteles = df_hoteles.drop(columns=['arrival_date'])

In [10]:
#Calcular la duración de la estancia
df_hoteles['stay_duration'] = df_hoteles['stays_in_weekend_nights'] + df_hoteles['stays_in_week_nights']
#Nueva columna
df_hoteles['only_week_nights'] = np.where(df_hoteles['stays_in_weekend_nights'] == 0, 1, 0)
#Nueva columna
df_hoteles['only_weekend_nights'] = np.where(df_hoteles['stays_in_week_nights'] == 0, 1, 0)

### Primer ejemplo de red neuronal con TensorFlow/Keras

In [11]:
#Checamos que tensorflow esté bien instalado
import tensorflow as tf
print("TF version:", tf.__version__)

from tensorflow import keras
print("Keras version:", keras.__version__)

# comprobar GPU (si corresponde)
tf.config.list_physical_devices('GPU')

2025-10-04 06:56:50.078793: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-04 06:56:50.556571: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.16.1
Keras version: 3.7.0


2025-10-04 06:56:54.786890: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

2025-10-04 06:56:55.024947: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-04 06:56:55.025027: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [13]:

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers, callbacks


In [14]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
import tensorflow as tf
tf.random.set_seed(RANDOM_STATE)

In [16]:

# --------------- Datos ---------------
# Selección de features: incluir todas menos target y arrival_date/ID (si existen)
exclude = {'children'}
features = [c for c in df_hoteles.columns if c not in exclude]
print('Número total de features candidatas:', len(features))
print(features)

target = 'children'

# X e y
X = df_hoteles[features].copy()
y = df_hoteles[target].astype(int).copy()


# revisión rápida de nulos
print('\nNulos por columna (top 10):')
print(X.isna().sum().sort_values(ascending=False).head(10))


Número total de features candidatas: 28
['hotel', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'average_daily_rate', 'required_car_parking_spaces', 'total_of_special_requests', 'arrival_month_sin', 'arrival_month_cos', 'stay_duration', 'only_week_nights', 'only_weekend_nights']

Nulos por columna (top 10):
hotel                          0
lead_time                      0
only_week_nights               0
stay_duration                  0
arrival_month_cos              0
arrival_month_sin              0
total_of_special_requests      0
required_car_parking_spaces    0
average_daily_rate             0
customer_type                  0
dtype: int64


In [17]:
# %%
# ---------------- split ----------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

print('\nTrain shape:', X_train.shape, 'Test shape:', X_test.shape)


Train shape: (42384, 28) Test shape: (10597, 28)


In [18]:
from sklearn.pipeline import Pipeline
# ---------------- preprocesamiento ----------------
# Detectar numéricas y categóricas (incluye int y float en numéricas)
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


print('\nNum columns:', len(num_cols), num_cols)
print('\nCat columns:', len(cat_cols), cat_cols)


# Pipelines: imputar -> escalar para numéricas; imputar -> onehot para categóricas
num_pipeline = Pipeline([('scaler', StandardScaler())])

cat_pipeline = Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer(transformers=[
                    ('num', num_pipeline, num_cols),
                    ('cat', cat_pipeline, cat_cols)])

# Ajustar transformador solo con TRAIN
X_train_pp = preprocessor.fit_transform(X_train)
X_test_pp = preprocessor.transform(X_test)


# guardar preprocessor para usar en producción
joblib.dump(preprocessor, 'preprocessor.joblib')


input_dim = X_train_pp.shape[1]
print('\nInput dim (n columnas tras preprocesado):', input_dim)


Num columns: 19 ['hotel', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'agent', 'company', 'days_in_waiting_list', 'average_daily_rate', 'total_of_special_requests', 'arrival_month_sin', 'arrival_month_cos', 'stay_duration', 'only_week_nights', 'only_weekend_nights']

Cat columns: 9 ['meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'required_car_parking_spaces']

Input dim (n columnas tras preprocesado): 75


In [19]:
# ---------------- construir modelo ----------------
def build_model(input_dim):
    model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') # salida binaria
    ])
    model.compile(optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')])
    return model


model = build_model(input_dim)

2025-10-04 07:02:41.393797: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-04 07:02:41.393969: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-04 07:02:41.393996: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-04 07:02:43.434912: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-10-04 07:02:43.434994: I external/local_xla/xla/stream_executor

In [67]:
# ---------------- manejar desbalance con class_weight ----------------
# classes = np.unique(y_train)
# class_weights_arr = compute_class_weight('balanced', classes=classes, y=y_train)
# class_weights = {k: class_weights_arr[i] for i,k in enumerate(classes)}
# print('\nclass_weights:', class_weights)

In [20]:
# ---------------- callbacks ----------------
# Usar extensión .keras para ModelCheckpoint (formato recomendado)
es = callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True)
mc = callbacks.ModelCheckpoint('best_model.keras', monitor='val_auc', mode='max', save_best_only=True)

In [21]:
# ---------------- entrenar ----------------
history = model.fit(
X_train_pp, y_train,
validation_split=0.15, # usa parte del train para validar
epochs=50,
batch_size=128,
#class_weight=class_weights,
callbacks=[es, mc],
verbose=2
)

Epoch 1/50


I0000 00:00:1759582983.805424    5357 service.cc:145] XLA service 0x7fbfd80068b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1759582983.805491    5357 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-04 07:03:03.840120: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-04 07:03:04.013473: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1759582986.211119    5357 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


282/282 - 8s - 29ms/step - accuracy: 0.8726 - auc: 0.7103 - loss: 0.3069 - val_accuracy: 0.9286 - val_auc: 0.8661 - val_loss: 0.1999
Epoch 2/50
282/282 - 1s - 4ms/step - accuracy: 0.9324 - auc: 0.8547 - loss: 0.2022 - val_accuracy: 0.9352 - val_auc: 0.8900 - val_loss: 0.1826
Epoch 3/50
282/282 - 1s - 4ms/step - accuracy: 0.9360 - auc: 0.8739 - loss: 0.1907 - val_accuracy: 0.9385 - val_auc: 0.9005 - val_loss: 0.1744
Epoch 4/50
282/282 - 1s - 4ms/step - accuracy: 0.9384 - auc: 0.8869 - loss: 0.1822 - val_accuracy: 0.9412 - val_auc: 0.9074 - val_loss: 0.1688
Epoch 5/50
282/282 - 1s - 4ms/step - accuracy: 0.9408 - auc: 0.8902 - loss: 0.1779 - val_accuracy: 0.9424 - val_auc: 0.9107 - val_loss: 0.1651
Epoch 6/50
282/282 - 1s - 4ms/step - accuracy: 0.9425 - auc: 0.8989 - loss: 0.1721 - val_accuracy: 0.9429 - val_auc: 0.9133 - val_loss: 0.1629
Epoch 7/50
282/282 - 1s - 4ms/step - accuracy: 0.9443 - auc: 0.9010 - loss: 0.1694 - val_accuracy: 0.9431 - val_auc: 0.9149 - val_loss: 0.1615
Epoch 8/5

In [22]:
# ---------------- evaluar ----------------
results = model.evaluate(X_test_pp, y_test, verbose=0)
print("Test loss, acc, auc:", results)

Test loss, acc, auc: [0.1572394073009491, 0.9451731443405151, 0.9230935573577881]


In [23]:
# ---------------- guardar modelo final (formato Keras native .keras) ----------------
model.save('mi_red_tf_keras.keras')
print('Modelo guardado en mi_red_tf_keras.keras')

Modelo guardado en mi_red_tf_keras.keras


### Modelo sobre conjunto de prueba

In [24]:
df_prueba = pd.read_csv('../../data/inputs//hoteles-prueba.csv')
preprocessor = joblib.load('preprocessor.joblib')
model = keras.models.load_model('mi_red_tf_keras.keras')

In [25]:
#Mismo proceso de preprocesamiento que en entrenamiento
# Copia de prueba
df_prueba = df_prueba.copy()

# Convertimos variable'hotel' a numérica
df_prueba['hotel'] = df_prueba['hotel'].map({'Resort_Hotel': 1, 'City_Hotel': 0})

# arrival_date a datetime
df_prueba['arrival_date'] = pd.to_datetime(df_prueba['arrival_date'], format='%Y-%m-%d')

# Extraer mes y codificación cíclica
df_prueba['arrival_month'] = df_prueba['arrival_date'].dt.month
df_prueba['arrival_month_sin'] = np.sin(2 * np.pi * df_prueba['arrival_month']/12)
df_prueba['arrival_month_cos'] = np.cos(2 * np.pi * df_prueba['arrival_month']/12)
df_prueba = df_prueba.drop(columns=['arrival_date', 'arrival_month'])

# Crear variables adicionales
df_prueba['stay_duration'] = df_prueba['stays_in_weekend_nights'] + df_prueba['stays_in_week_nights']

df_prueba['only_week_nights'] = np.where(df_prueba['stays_in_weekend_nights'] == 0, 1, 0)
df_prueba['only_weekend_nights'] = np.where(df_prueba['stays_in_week_nights'] == 0, 1, 0)

# Agrupar country como hiciste antes
df_prueba['country'] = df_prueba['country'].where(df_prueba['country'].isin(top10_list), 'OTROS')


In [26]:
#Seleccionar features 
X_prueba = df_prueba[features].copy()
#Transformar con el preprocesador guardado
X_prueba_pp = preprocessor.transform(X_prueba)
# Obtener las probabilidades de la clase positiva
probs = model.predict(X_prueba_pp).ravel()


694/694 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [27]:
print(probs[:5])

[nan nan nan nan nan]


In [ ]:
df_out = pd.DataFrame({'ID': df_prueba['id'], 'prob': probs})
df_out.to_csv('../../data/soutputs/Salida_{}.csv'.format(pd.Timestamp.now().strftime('%y%m%d')), index=False)